In [1]:
import sys
import random
import numpy as np
import torch

SEED = 42

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime in Colab, then rerun this cell.")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

device = torch.device("cuda")
gpu = torch.cuda.get_device_properties(device)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"GPU: {gpu.name}")
print(f"GPU memory: {gpu.total_memory / 1024**3:.1f} GB")
print(f"Random seed: {SEED}")

Python: 3.13.16
NumPy: 2.4.6
PyTorch: 2.14.1+cu130
CUDA version: 13.0
GPU: Tesla T4
GPU memory: 14.6 GB
Random seed: 42


In [2]:
%pip install -q "datasets>=4,<5" "pandas>=2.2,<4" "scikit-learn>=1.6,<2"

import pandas as pd
import datasets
import sklearn

from datasets import load_dataset
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", None)

print(f"Datasets: {datasets.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print("Data-preparation libraries are ready.")

Datasets: 4.8.5
Pandas: 2.2.3
Scikit-learn: 1.6.1
Data-preparation libraries are ready.


In [3]:
from huggingface_hub import HfApi

DATASET_NAME = "cnamuangtoun/resume-job-description-fit"
DATASET_REVISION = HfApi().dataset_info(DATASET_NAME).sha

LABELS = ["Good Fit", "Potential Fit", "No Fit"]
REQUIRED_COLUMNS = ["resume_text", "job_description_text", "label"]

dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)

assert {"train", "test"}.issubset(dataset.keys())

for split_name in ["train", "test"]:
    assert set(REQUIRED_COLUMNS).issubset(dataset[split_name].column_names)

train_raw_df = dataset["train"].to_pandas()[REQUIRED_COLUMNS].copy()
test_raw_df = dataset["test"].to_pandas()[REQUIRED_COLUMNS].copy()

for dataframe in [train_raw_df, test_raw_df]:
    assert set(dataframe["label"].dropna()) == set(LABELS)

label_counts = pd.DataFrame({
    "Original training pool": train_raw_df["label"].value_counts(),
    "Official test": test_raw_df["label"].value_counts()
}).reindex(LABELS).fillna(0).astype(int)

label_counts["Total"] = label_counts.sum(axis=1)

print(f"Dataset: {DATASET_NAME}")
print(f"Revision: {DATASET_REVISION}")
print(f"Original training rows: {len(train_raw_df):,}")
print(f"Official test rows: {len(test_raw_df):,}")
print(f"Columns: {REQUIRED_COLUMNS}")

display(label_counts)

train.csv: reconstructing file:   0%|          |  0.00B / 53.4MB            

train.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 15.2MB            

test.csv: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset: cnamuangtoun/resume-job-description-fit
Revision: 08978e21714984bb417547d2c0f9b477f5298163
Original training rows: 6,241
Official test rows: 1,759
Columns: ['resume_text', 'job_description_text', 'label']


,Original training pool,Official test,Total
label,,,
Good Fit,1542,458,2000
Potential Fit,1556,444,2000
No Fit,3143,857,4000


In [4]:
PAIR_COLUMNS = ["resume_text", "job_description_text"]

quality_rows = []

for split_name, dataframe in [
    ("Original training", train_raw_df),
    ("Official test", test_raw_df)
]:
    for column in REQUIRED_COLUMNS:
        quality_rows.append({
            "Split": split_name,
            "Column": column,
            "Missing": int(dataframe[column].isna().sum()),
            "Empty": int(
                dataframe[column].astype("string").str.strip().eq("").sum()
            )
        })

quality_report = pd.DataFrame(quality_rows)
display(quality_report)

assert quality_report[["Missing", "Empty"]].to_numpy().sum() == 0

pair_label_counts = train_raw_df.groupby(PAIR_COLUMNS)["label"].transform("nunique")
conflicting_mask = pair_label_counts > 1

conflicting_groups = (
    train_raw_df.loc[conflicting_mask, PAIR_COLUMNS].drop_duplicates().shape[0]
)

consistent_train_df = train_raw_df.loc[~conflicting_mask].copy()

train_pool_df = consistent_train_df.drop_duplicates(
    subset=PAIR_COLUMNS, keep="first"
).copy()

train_pool_df["source_row_id"] = train_pool_df.index
train_pool_df = train_pool_df.reset_index(drop=True)

duplicate_summary = pd.DataFrame({
    "Check": [
        "Original training rows",
        "Conflicting pair groups",
        "Conflicting rows removed",
        "Extra same-label copies removed",
        "Remaining training pool",
        "Official test duplicate copies"
    ],
    "Count": [
        len(train_raw_df),
        conflicting_groups,
        int(conflicting_mask.sum()),
        len(consistent_train_df) - len(train_pool_df),
        len(train_pool_df),
        int(test_raw_df.duplicated(PAIR_COLUMNS).sum())
    ]
})

display(duplicate_summary)

assert not train_pool_df.duplicated(PAIR_COLUMNS).any()
assert not test_raw_df.duplicated(PAIR_COLUMNS).any()

,Split,Column,Missing,Empty
0,Original training,resume_text,0,0
1,Original training,job_description_text,0,0
2,Original training,label,0,0
3,Official test,resume_text,0,0
4,Official test,job_description_text,0,0
5,Official test,label,0,0


,Check,Count
0,Original training rows,6241
1,Conflicting pair groups,6
2,Conflicting rows removed,12
3,Extra same-label copies removed,1
4,Remaining training pool,6228
5,Official test duplicate copies,0


In [5]:
import re
import hashlib
import unicodedata

EMAIL_PATTERN = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
URL_PATTERN = re.compile(r"(?:https?://|www\.)\S+", re.IGNORECASE)
PHONE_PATTERN = re.compile(r"(?<!\w)(?:\+?\d[\d\s().-]{8,}\d)(?!\w)")


def prepare_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = EMAIL_PATTERN.sub(" email ", text)
    text = URL_PATTERN.sub(" url ", text)
    text = PHONE_PATTERN.sub(
        lambda match: " phone "
        if sum(character.isdigit() for character in match.group()) >= 10
        else match.group(),
        text
    )
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[^\S\n]+", " ", text)
    return re.sub(r"\n\s*\n+", "\n", text).strip()


def text_group_id(text):
    normalized = " ".join(text.casefold().split())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


assert prepare_text("Skills: C++, C#, .NET") == "Skills: C++, C#, .NET"
print("Text preparation functions are ready.")

Text preparation functions are ready.


In [6]:
def prepare_frame(dataframe):
    result = dataframe.copy()
    result["resume_model_text"] = result["resume_text"].map(prepare_text)
    result["job_model_text"] = result["job_description_text"].map(prepare_text)
    result["resume_group"] = result["resume_model_text"].map(text_group_id)
    result["job_group"] = result["job_model_text"].map(text_group_id)
    return result


train_pool_df = prepare_frame(train_pool_df)
test_prepared_df = prepare_frame(test_raw_df)
test_prepared_df["source_row_id"] = test_raw_df.index

MODEL_PAIR_COLUMNS = ["resume_group", "job_group"]
preparation_rows = []

for split_name, dataframe in [
    ("Training pool", train_pool_df),
    ("Official test", test_prepared_df)
]:
    preparation_rows.append({
        "Split": split_name,
        "Rows": len(dataframe),
        "Empty resumes": int(dataframe["resume_model_text"].eq("").sum()),
        "Empty jobs": int(dataframe["job_model_text"].eq("").sum()),
        "Unique resumes": dataframe["resume_group"].nunique(),
        "Unique jobs": dataframe["job_group"].nunique(),
        "Normalized duplicate copies": int(
            dataframe.duplicated(MODEL_PAIR_COLUMNS).sum()
        )
    })

preparation_report = pd.DataFrame(preparation_rows)
display(preparation_report)

assert preparation_report[["Empty resumes", "Empty jobs"]].to_numpy().sum() == 0

,Split,Rows,Empty resumes,Empty jobs,Unique resumes,Unique jobs,Normalized duplicate copies
0,Training pool,6228,0,0,642,280,0
1,Official test,1759,0,0,477,71,0


In [7]:
overlap_rows = []

for unit, columns in [
    ("Resume", ["resume_group"]),
    ("Job", ["job_group"]),
    ("Resume-job pair", MODEL_PAIR_COLUMNS)
]:
    training_keys = set(
        train_pool_df[columns].itertuples(index=False, name=None)
    )
    test_keys = list(
        test_prepared_df[columns].itertuples(index=False, name=None)
    )

    overlap_rows.append({
        "Unit": unit,
        "Shared unique values": len(training_keys.intersection(test_keys)),
        "Test rows with overlap": sum(key in training_keys for key in test_keys)
    })

overlap_report = pd.DataFrame(overlap_rows).set_index("Unit")
display(overlap_report)

assert overlap_report.loc["Job", "Shared unique values"] == 0
assert overlap_report.loc["Resume-job pair", "Shared unique values"] == 0
assert test_prepared_df[REQUIRED_COLUMNS].equals(test_raw_df)
assert not train_pool_df.duplicated(MODEL_PAIR_COLUMNS).any()
assert not test_prepared_df.duplicated(MODEL_PAIR_COLUMNS).any()

print("Test jobs and pairs are separate. Original test records are unchanged.")

,Shared unique values,Test rows with overlap
Unit,,
Resume,476,1758
Job,0,0
Resume-job pair,0,0


Test jobs and pairs are separate. Original test records are unchanged.


In [8]:
splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

train_indices, validation_indices = next(
    splitter.split(
        X=train_pool_df,
        y=train_pool_df["label"],
        groups=train_pool_df["job_group"]
    )
)

train_df = train_pool_df.iloc[train_indices].reset_index(drop=True)
validation_df = train_pool_df.iloc[validation_indices].reset_index(drop=True)
test_df = test_prepared_df.copy()

split_frames = {
    "Training": train_df,
    "Validation": validation_df,
    "Official test": test_df
}

split_counts = pd.DataFrame({
    name: dataframe["label"].value_counts()
    for name, dataframe in split_frames.items()
}).reindex(LABELS).fillna(0).astype(int).T

split_counts["Total"] = split_counts.sum(axis=1)
split_counts["Unique jobs"] = [
    dataframe["job_group"].nunique()
    for dataframe in split_frames.values()
]

display(split_counts)

class_percentages = (
    split_counts[LABELS]
    .div(split_counts["Total"], axis=0)
    .mul(100)
    .round(2)
)
display(class_percentages)

print(f"Validation share: {len(validation_df) / len(train_pool_df):.1%}")

label,Good Fit,Potential Fit,No Fit,Total,Unique jobs
Training,1258,1223,2600,5081,229
Validation,280,331,536,1147,51
Official test,458,444,857,1759,71


label,Good Fit,Potential Fit,No Fit
Training,24.76,24.07,51.17
Validation,24.41,28.86,46.73
Official test,26.04,25.24,48.72


Validation share: 18.4%


In [9]:
from itertools import combinations

split_overlap_rows = []

for (first_name, first_df), (second_name, second_df) in combinations(
    split_frames.items(), 2
):
    first_pairs = set(
        first_df[MODEL_PAIR_COLUMNS].itertuples(index=False, name=None)
    )
    second_pairs = set(
        second_df[MODEL_PAIR_COLUMNS].itertuples(index=False, name=None)
    )

    split_overlap_rows.append({
        "Comparison": f"{first_name} / {second_name}",
        "Shared jobs": len(
            set(first_df["job_group"]) & set(second_df["job_group"])
        ),
        "Shared resumes": len(
            set(first_df["resume_group"]) & set(second_df["resume_group"])
        ),
        "Shared pairs": len(first_pairs & second_pairs)
    })

split_overlap_report = pd.DataFrame(split_overlap_rows)
display(split_overlap_report)

assert split_overlap_report["Shared jobs"].eq(0).all()
assert split_overlap_report["Shared pairs"].eq(0).all()

training_ids = set(train_df["source_row_id"])
validation_ids = set(validation_df["source_row_id"])

assert training_ids.isdisjoint(validation_ids)
assert training_ids | validation_ids == set(train_pool_df["source_row_id"])
assert len(train_df) + len(validation_df) == len(train_pool_df)
assert test_df[REQUIRED_COLUMNS].equals(test_raw_df)

for dataframe in split_frames.values():
    assert set(dataframe["label"]) == set(LABELS)
    assert not dataframe.duplicated(MODEL_PAIR_COLUMNS).any()

print("All split checks passed. No jobs or pairs are shared across splits.")

,Comparison,Shared jobs,Shared resumes,Shared pairs
0,Training / Validation,0,429,0
1,Training / Official test,0,473,0
2,Validation / Official test,0,357,0


All split checks passed. No jobs or pairs are shared across splits.


In [10]:
import json
from pathlib import Path
from importlib.metadata import version

OUTPUT_DIR = Path("/content/m3_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

split_manifest = {
    "dataset": DATASET_NAME,
    "dataset_revision": DATASET_REVISION,
    "seed": SEED,
    "splitter": "StratifiedGroupKFold",
    "n_splits": 5,
    "validation_fold": 0,
    "group_column": "job_group",
    "labels": LABELS,
    "python_version": sys.version.split()[0],
    "package_versions": {
        package: version(package)
        for package in ["numpy", "pandas", "datasets", "scikit-learn", "torch"]
    },
    "source_splits": {
        "Training": "train",
        "Validation": "train",
        "Official test": "test"
    },
    "source_row_ids": {
        name: dataframe["source_row_id"].astype(int).tolist()
        for name, dataframe in split_frames.items()
    }
}

manifest_path = OUTPUT_DIR / "split_manifest.json"
manifest_path.write_text(json.dumps(split_manifest, indent=2), encoding="utf-8")

split_counts.to_csv(OUTPUT_DIR / "split_counts.csv", index_label="Split")
split_overlap_report.to_csv(OUTPUT_DIR / "split_overlap.csv", index=False)

print(f"Saved files in: {OUTPUT_DIR}")

for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size:,} bytes")

Saved files in: /content/m3_artifacts
split_counts.csv: 152 bytes
split_manifest.json: 94,346 bytes
split_overlap.csv: 149 bytes


In [11]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

saved_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

assert saved_manifest["dataset_revision"] == DATASET_REVISION

source_frames = {
    "train": train_pool_df.set_index("source_row_id", drop=False),
    "test": test_prepared_df.set_index("source_row_id", drop=False)
}

verification_rows = []

for name, expected_df in split_frames.items():
    source_name = saved_manifest["source_splits"][name]
    row_ids = saved_manifest["source_row_ids"][name]

    assert len(row_ids) == len(set(row_ids))

    reconstructed_df = (
        source_frames[source_name].loc[row_ids].reset_index(drop=True)
    )

    verification_rows.append({
        "Split": name,
        "Rows": len(reconstructed_df),
        "Exact match": reconstructed_df.equals(expected_df)
    })

verification_report = pd.DataFrame(verification_rows)
display(verification_report)

assert verification_report["Exact match"].all()

archive_path = Path("/content/m3_data_setup.zip")

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for filename in ["split_manifest.json", "split_counts.csv", "split_overlap.csv"]:
        archive.write(OUTPUT_DIR / filename, arcname=filename)

print("All saved splits reproduce exactly. Downloading the setup archive.")
files.download(str(archive_path))

,Split,Rows,Exact match
0,Training,5081,True
1,Validation,1147,True
2,Official test,1759,True


All saved splits reproduce exactly. Downloading the setup archive.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>